In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.min_rows', None)

In [2]:
promotions = pd.read_csv('datasets/promotions/promotions.csv', parse_dates=['start_date','end_date'])
orders = pd.read_csv('datasets/promotions/orders.csv', parse_dates=['order_date'])

In [3]:
promotions

,promo_id,promo_name,start_date,end_date
0,BF_2023,Black Friday,2023-11-24,2023-11-29
1,NY_2024,New Year Sale,2024-01-01,2024-01-07
2,SB_2024,Summer Blowout,2024-07-15,2024-07-31
3,BF_2024,Black Friday,2024-11-25,2024-11-30
4,NY_2025,New Year Sale,2025-01-01,2025-01-07
5,SC_2025,Spring Clearance,2025-03-10,2025-03-20


In [4]:
orders.head()

,order_id,order_date,order_quantity
0,1,2023-05-01,2
1,2,2023-05-01,3
2,3,2023-05-01,1
3,4,2023-05-01,5
4,5,2023-05-01,2


# Method 1: Using merge_asof

In [6]:
df = pd.merge_asof(
    orders.sort_values('order_date'),
    promotions.sort_values('start_date'),
    left_on='order_date',
    right_on='start_date',
    direction='backward'
)
df.loc[df['order_date'] > df['end_date'], 'promo_id'] = np.nan
df[['order_id','order_date','order_quantity','promo_id']]

,order_id,order_date,order_quantity,promo_id
0,1,2023-05-01,2,NaN
1,2,2023-05-01,3,NaN
2,3,2023-05-01,1,NaN
3,4,2023-05-01,5,NaN
4,5,2023-05-01,2,NaN
5,6,2023-05-02,2,NaN
6,7,2023-05-02,4,NaN
7,8,2023-05-02,4,NaN
8,9,2023-05-02,3,NaN
9,10,2023-05-03,2,NaN


In [7]:
df['promo_id'].isna().sum()

1916

# Method 2: Using IntervalIndex

In [8]:
intervals = pd.IntervalIndex.from_arrays(
    promotions['start_date'], promotions['end_date'], closed='both'
)
idx = intervals.get_indexer(orders["order_date"])
data2_matched = promotions.iloc[idx].reset_index(drop=True)
data2_matched.index = orders.index

no_match = idx == -1
df = orders.copy()
df['promo_id'] = data2_matched['promo_id'].where(~pd.Series(no_match, index=orders.index))
df[['start_date', 'end_date']] = data2_matched[['start_date', 'end_date']].where(
    ~pd.Series(no_match, index=orders.index)
)

df

,order_id,order_date,order_quantity,promo_id,start_date,end_date
0,1,2023-05-01,2,NaN,NaT,NaT
1,2,2023-05-01,3,NaN,NaT,NaT
2,3,2023-05-01,1,NaN,NaT,NaT
3,4,2023-05-01,5,NaN,NaT,NaT
4,5,2023-05-01,2,NaN,NaT,NaT
5,6,2023-05-02,2,NaN,NaT,NaT
6,7,2023-05-02,4,NaN,NaT,NaT
7,8,2023-05-02,4,NaN,NaT,NaT
8,9,2023-05-02,3,NaN,NaT,NaT
9,10,2023-05-03,2,NaN,NaT,NaT


## How many orders were placed outside of promotional periods?

In [9]:
df['promo_id'].isna().sum()

1916